# NB_Gold_Model
Builds the reporting model from the **current** Silver rows. Safe to re-run (`CREATE OR REPLACE`).

| Object | Type |
|---|---|
| `gold.dim_patient`, `gold.dim_date` | dimension tables |
| `gold.fact_encounter`, `gold.fact_observation`, `gold.fact_condition` | fact tables (keys: `patient_id`, `encounter_id`) |
| `gold.vw_patient_summary`, `gold.vw_top_conditions`, `gold.vw_observation_summary`, `gold.vw_encounters_by_day` | reporting views |
| `meta.vw_reconciliation` | API total vs raw vs bronze vs silver counts per resource and day |

In [1]:
# SAFE GOLD TABLE INSPECTION
# No tables or data will be modified.

gold_tables = [
    "gold.dim_patient",
    "gold.fact_encounter",
    "gold.fact_observation",
    "gold.fact_condition",
    "gold.dim_date"
]

for table_name in gold_tables:

    print(f"\n========== {table_name} ==========")

    if spark.catalog.tableExists(table_name):

        df = spark.table(table_name)

        print("Status: EXISTS")
        print("Total records:", df.count())
        df.printSchema()

    else:
        print("Status: DOES NOT EXIST")

StatementMeta(, fece9998-cbed-42df-ae4e-aaeaee85b9c8, 3, Finished, Available, Finished, False)


========== gold.dim_patient ==========
Status: EXISTS
Total records: 1052
root
 |-- patient_id: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- birth_date: string (nullable = true)
 |-- family_name: string (nullable = true)
 |-- given_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- valid_from: timestamp (nullable = true)
 |-- gold_loaded_at: timestamp (nullable = true)


========== gold.fact_encounter ==========
Status: EXISTS
Total records: 2483
root
 |-- encounter_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- class_code: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- period_start: string (nullable = true)
 |-- period_end: string (nullable = true)
 |-- gold_loaded_at: timestamp (nullable = true)


========== gold.fact_observation ==========
Status: EXISTS
Total records: 39309


In [3]:
# ============================================
# CELL 2: IMPORTS AND SILVER SOURCE INSPECTION
# ============================================

from pyspark.sql import functions as F
from pyspark.sql import Window
from datetime import datetime, timezone
from delta.tables import DeltaTable

silver_tables = [
    "silver.patient",
    "silver.encounter",
    "silver.observation",
    "silver.condition"
]

for table_name in silver_tables:

    print(f"\n========== {table_name} ==========")

    if spark.catalog.tableExists(table_name):

        df = spark.table(table_name)

        print("Status: EXISTS")
        print("Total records:", df.count())

        if "is_current" in df.columns:
            print(
                "Current records:",
                df.filter(F.col("is_current") == True).count()
            )

            print(
                "Historical records:",
                df.filter(F.col("is_current") == False).count()
            )

        print("Schema:")
        df.printSchema()

    else:
        print("Status: DOES NOT EXIST")

StatementMeta(, fece9998-cbed-42df-ae4e-aaeaee85b9c8, 5, Finished, Available, Finished, False)


========== silver.patient ==========
Status: EXISTS
Total records: 1052
Current records: 1052
Historical records: 0
Schema:
root
 |-- id: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- birth_date: string (nullable = true)
 |-- family_name: string (nullable = true)
 |-- given_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- source_last_updated: timestamp (nullable = true)
 |-- extraction_timestamp: timestamp (nullable = true)
 |-- api_url_or_params: string (nullable = true)
 |-- ingest_date: string (nullable = true)
 |-- hash_diff: string (nullable = true)
 |-- effective_from: timestamp (nullable = true)
 |-- effective_to: timestamp (nullable = true)
 |-- is_current: boolean (nullable = true)
 |-- run_id: string (nullable = true)


========== silver.encounter ==========
Status: EXISTS
Total records: 7288
Current records: 2483
Historical records: 4805
Schema:
root
 |--

In [4]:
# ============================================
# CELL 3: GOLD CONFIGURATION
# ============================================

from datetime import datetime, timezone

gold_run_time = datetime.now(timezone.utc).replace(tzinfo=None)

# Silver source tables
patient_src = "silver.patient"
encounter_src = "silver.encounter"
observation_src = "silver.observation"
condition_src = "silver.condition"

# Gold target tables
patient_tgt = "gold.dim_patient"
encounter_tgt = "gold.fact_encounter"
observation_tgt = "gold.fact_observation"
condition_tgt = "gold.fact_condition"
date_tgt = "gold.dim_date"

print("Gold configuration loaded successfully.")
print("Gold processing time:", gold_run_time)

print("\nSilver sources:")
print(patient_src)
print(encounter_src)
print(observation_src)
print(condition_src)

print("\nGold targets:")
print(patient_tgt)
print(encounter_tgt)
print(observation_tgt)
print(condition_tgt)
print(date_tgt)

StatementMeta(, fece9998-cbed-42df-ae4e-aaeaee85b9c8, 6, Finished, Available, Finished, False)

Gold configuration loaded successfully.
Gold processing time: 2026-11-06 17:08:44.398925

Silver sources:
silver.patient
silver.encounter
silver.observation
silver.condition

Gold targets:
gold.dim_patient
gold.fact_encounter
gold.fact_observation
gold.fact_condition
gold.dim_date


In [5]:
# ============================================
# CELL 4: READ CURRENT SILVER DATA
# ============================================

silver_patient = (
    spark.table(patient_src)
    .filter(F.col("is_current") == True)
)

silver_encounter = (
    spark.table(encounter_src)
    .filter(F.col("is_current") == True)
)

silver_observation = (
    spark.table(observation_src)
    .filter(F.col("is_current") == True)
)

silver_condition = (
    spark.table(condition_src)
    .filter(F.col("is_current") == True)
)

print("Current Silver records loaded successfully.")

print("Patient:", silver_patient.count())
print("Encounter:", silver_encounter.count())
print("Observation:", silver_observation.count())
print("Condition:", silver_condition.count())

StatementMeta(, fece9998-cbed-42df-ae4e-aaeaee85b9c8, 7, Finished, Available, Finished, False)

Current Silver records loaded successfully.
Patient: 1052
Encounter: 2483
Observation: 39309
Condition: 1611


In [6]:
# ============================================
# CELL 5: BUILD GOLD PATIENT DIMENSION
# ============================================

gold_patient = (
    silver_patient
    .select(
        F.col("id").alias("patient_id"),
        "gender",
        "birth_date",
        "family_name",
        "given_name",
        "city",
        "state",
        "country",
        F.col("effective_from").alias("valid_from")
    )
    .withColumn(
        "gold_loaded_at",
        F.lit(gold_run_time).cast("timestamp")
    )
    .dropDuplicates(["patient_id"])
)

print("Gold Patient dimension prepared.")
print("Patient dimension records:", gold_patient.count())

print("Gold Patient schema:")
gold_patient.printSchema()

display(gold_patient.limit(10))

StatementMeta(, fece9998-cbed-42df-ae4e-aaeaee85b9c8, 8, Finished, Available, Finished, False)

Gold Patient dimension prepared.
Patient dimension records: 1052
Gold Patient schema:
root
 |-- patient_id: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- birth_date: string (nullable = true)
 |-- family_name: string (nullable = true)
 |-- given_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- valid_from: timestamp (nullable = true)
 |-- gold_loaded_at: timestamp (nullable = false)



SynapseWidget(Synapse.DataFrame, cffbe63c-7b5a-4571-b31d-fee13452ee9d)

In [7]:
# ============================================
# CELL 6: BUILD GOLD ENCOUNTER FACT
# ============================================

gold_encounter = (
    silver_encounter
    .select(
        F.col("id").alias("encounter_id"),
        "patient_id",
        "status",
        "class_code",
        F.to_date("period_start").alias("start_date"),
        "period_start",
        "period_end"
    )
    .withColumn(
        "gold_loaded_at",
        F.lit(gold_run_time).cast("timestamp")
    )
    .dropDuplicates(["encounter_id"])
)

print("Gold Encounter fact prepared.")
print("Encounter fact records:", gold_encounter.count())

print("Gold Encounter schema:")
gold_encounter.printSchema()

display(gold_encounter.limit(10))

StatementMeta(, fece9998-cbed-42df-ae4e-aaeaee85b9c8, 9, Finished, Available, Finished, False)

Gold Encounter fact prepared.
Encounter fact records: 2483
Gold Encounter schema:
root
 |-- encounter_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- class_code: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- period_start: string (nullable = true)
 |-- period_end: string (nullable = true)
 |-- gold_loaded_at: timestamp (nullable = false)



SynapseWidget(Synapse.DataFrame, 9f3011ca-1a9f-48f7-84db-ca275156e7aa)

In [8]:
# ============================================
# CELL 7: BUILD GOLD OBSERVATION FACT
# ============================================

gold_observation = (
    silver_observation
    .select(
        F.col("id").alias("observation_id"),
        "patient_id",
        "encounter_id",
        "code",
        "code_display",
        "value_number",
        "value_unit",
        F.to_date("effective_datetime").alias("effective_date")
    )
    .withColumn(
        "gold_loaded_at",
        F.lit(gold_run_time).cast("timestamp")
    )
    .dropDuplicates(["observation_id"])
)

print("Gold Observation fact prepared.")
print("Observation fact records:", gold_observation.count())

print("Gold Observation schema:")
gold_observation.printSchema()

display(gold_observation.limit(10))

StatementMeta(, fece9998-cbed-42df-ae4e-aaeaee85b9c8, 10, Finished, Available, Finished, False)

Gold Observation fact prepared.
Observation fact records: 39309
Gold Observation schema:
root
 |-- observation_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- encounter_id: string (nullable = true)
 |-- code: string (nullable = true)
 |-- code_display: string (nullable = true)
 |-- value_number: double (nullable = true)
 |-- value_unit: string (nullable = true)
 |-- effective_date: date (nullable = true)
 |-- gold_loaded_at: timestamp (nullable = false)



SynapseWidget(Synapse.DataFrame, c32a4c03-84ae-47e7-b41c-2e738e2bfa1b)

In [9]:
# ============================================
# CELL 8: BUILD GOLD CONDITION FACT
# ============================================

gold_condition = (
    silver_condition
    .select(
        F.col("id").alias("condition_id"),
        "patient_id",
        "encounter_id",
        "code",
        "code_display",
        "clinical_status",
        F.to_date("onset_datetime").alias("onset_date")
    )
    .withColumn(
        "gold_loaded_at",
        F.lit(gold_run_time).cast("timestamp")
    )
    .dropDuplicates(["condition_id"])
)

print("Gold Condition fact prepared.")
print("Condition fact records:", gold_condition.count())

print("Gold Condition schema:")
gold_condition.printSchema()

display(gold_condition.limit(10))

StatementMeta(, fece9998-cbed-42df-ae4e-aaeaee85b9c8, 11, Finished, Available, Finished, False)

Gold Condition fact prepared.
Condition fact records: 1611
Gold Condition schema:
root
 |-- condition_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- encounter_id: string (nullable = true)
 |-- code: string (nullable = true)
 |-- code_display: string (nullable = true)
 |-- clinical_status: string (nullable = true)
 |-- onset_date: date (nullable = true)
 |-- gold_loaded_at: timestamp (nullable = false)



SynapseWidget(Synapse.DataFrame, 6a09ebfa-48e0-4e86-bb01-34ff4a9568b5)

In [10]:
# ============================================
# CELL 9: BUILD GOLD DATE DIMENSION
# ============================================

date_values = (
    gold_encounter.select(F.col("start_date").alias("date_key"))
    .union(gold_observation.select(F.col("effective_date").alias("date_key")))
    .union(gold_condition.select(F.col("onset_date").alias("date_key")))
    .filter(F.col("date_key").isNotNull())
    .distinct()
)

gold_date = (
    date_values
    .withColumn("year", F.year("date_key"))
    .withColumn("month", F.month("date_key"))
    .withColumn("month_name", F.date_format("date_key", "MMMM"))
    .withColumn("day_of_week", F.dayofweek("date_key"))
    .withColumn(
        "gold_loaded_at",
        F.lit(gold_run_time).cast("timestamp")
    )
)

print("Gold Date dimension prepared.")
print("Date dimension records:", gold_date.count())

print("Gold Date schema:")
gold_date.printSchema()

display(gold_date.orderBy("date_key").limit(10))

StatementMeta(, fece9998-cbed-42df-ae4e-aaeaee85b9c8, 12, Finished, Available, Finished, False)

Gold Date dimension prepared.
Date dimension records: 1826
Gold Date schema:
root
 |-- date_key: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- month_name: string (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- gold_loaded_at: timestamp (nullable = false)



SynapseWidget(Synapse.DataFrame, 8be484d8-c045-49af-95fe-e8588fba7b91)

In [11]:
# ============================================
# CELL 10: VALIDATE GOLD DATA BEFORE LOADING
# ============================================

gold_checks = [
    (gold_patient, patient_tgt, "patient_id"),
    (gold_encounter, encounter_tgt, "encounter_id"),
    (gold_observation, observation_tgt, "observation_id"),
    (gold_condition, condition_tgt, "condition_id"),
    (gold_date, date_tgt, "date_key")
]

for source_df, target_table, key_column in gold_checks:

    print(f"\n========== {target_table} ==========")

    source_count = source_df.count()

    duplicate_count = (
        source_df.groupBy(key_column)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    null_count = source_df.filter(
        F.col(key_column).isNull()
    ).count()

    target_columns = set(spark.table(target_table).columns)
    source_columns = set(source_df.columns)

    missing_columns = source_columns - target_columns
    extra_columns = target_columns - source_columns

    print("Prepared records:", source_count)
    print("Duplicate IDs:", duplicate_count)
    print("Null IDs:", null_count)
    print("Missing target columns:", sorted(missing_columns))
    print("Additional target columns:", sorted(extra_columns))

    if duplicate_count > 0 or null_count > 0:
        raise RuntimeError(f"Duplicate or null keys found in {target_table}")

    if missing_columns or extra_columns:
        raise RuntimeError(f"Schema mismatch in {target_table}")

print("\nAll Gold datasets passed validation.")

StatementMeta(, fece9998-cbed-42df-ae4e-aaeaee85b9c8, 13, Finished, Available, Finished, False)


========== gold.dim_patient ==========
Prepared records: 1052
Duplicate IDs: 0
Null IDs: 0
Missing target columns: []
Additional target columns: []

========== gold.fact_encounter ==========
Prepared records: 2483
Duplicate IDs: 0
Null IDs: 0
Missing target columns: []
Additional target columns: []

========== gold.fact_observation ==========
Prepared records: 39309
Duplicate IDs: 0
Null IDs: 0
Missing target columns: []
Additional target columns: []

========== gold.fact_condition ==========
Prepared records: 1611
Duplicate IDs: 0
Null IDs: 0
Missing target columns: []
Additional target columns: []

========== gold.dim_date ==========
Prepared records: 1826
Duplicate IDs: 0
Null IDs: 0
Missing target columns: []
Additional target columns: []

All Gold datasets passed validation.


In [12]:
# ============================================
# CELL 11: GOLD UPSERT
# ============================================

def merge_gold(source_df, target_table, key_column):

    target = DeltaTable.forName(spark, target_table)

    compare_columns = [
        c for c in source_df.columns
        if c != "gold_loaded_at"
    ]

    change_condition = " OR ".join([
        f"NOT (t.`{c}` <=> s.`{c}`)"
        for c in compare_columns
    ])

    (
        target.alias("t")
        .merge(
            source_df.alias("s"),
            f"t.`{key_column}` = s.`{key_column}`"
        )
        .whenMatchedUpdateAll(condition=change_condition)
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"{target_table} upsert completed.")


merge_gold(gold_patient, patient_tgt, "patient_id")
merge_gold(gold_encounter, encounter_tgt, "encounter_id")
merge_gold(gold_observation, observation_tgt, "observation_id")
merge_gold(gold_condition, condition_tgt, "condition_id")
merge_gold(gold_date, date_tgt, "date_key")

print("\nAll Gold tables loaded successfully.")

StatementMeta(, fece9998-cbed-42df-ae4e-aaeaee85b9c8, 14, Finished, Available, Finished, False)

gold.dim_patient upsert completed.
gold.fact_encounter upsert completed.
gold.fact_observation upsert completed.
gold.fact_condition upsert completed.
gold.dim_date upsert completed.

All Gold tables loaded successfully.


In [ ]:
# ============================================
# REPORTING VIEWS
# ============================================
spark.sql("""
CREATE OR REPLACE VIEW gold.vw_encounters_per_patient AS
SELECT p.patient_id, p.gender, count(e.encounter_id) AS encounter_count
FROM gold.dim_patient p
LEFT JOIN gold.fact_encounter e ON p.patient_id = e.patient_id
GROUP BY p.patient_id, p.gender
""")

spark.sql("""
CREATE OR REPLACE VIEW gold.vw_top_conditions AS
SELECT code, code_display, count(*) AS cases, count(DISTINCT patient_id) AS patients
FROM gold.fact_condition
GROUP BY code, code_display
""")

spark.sql("""
CREATE OR REPLACE VIEW gold.vw_observation_summary AS
SELECT code, code_display, value_unit, count(*) AS readings,
       round(avg(value_number), 2) AS avg_value
FROM gold.fact_observation
GROUP BY code, code_display, value_unit
""")

display(spark.sql("SELECT * FROM gold.vw_top_conditions ORDER BY cases DESC LIMIT 10"))

In [13]:
# ============================================
# CELL 12: VERIFY GOLD TABLES AFTER UPSERT
# ============================================

gold_tables = [
    "gold.dim_patient",
    "gold.fact_encounter",
    "gold.fact_observation",
    "gold.fact_condition",
    "gold.dim_date"
]

for table_name in gold_tables:

    print(f"\n========== {table_name} ==========")

    df = spark.table(table_name)

    print("Total records:", df.count())
    print("Schema:")
    df.printSchema()

print("\nGold table verification completed.")

StatementMeta(, fece9998-cbed-42df-ae4e-aaeaee85b9c8, 15, Finished, Available, Finished, False)


========== gold.dim_patient ==========
Total records: 1052
Schema:
root
 |-- patient_id: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- birth_date: string (nullable = true)
 |-- family_name: string (nullable = true)
 |-- given_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- valid_from: timestamp (nullable = true)
 |-- gold_loaded_at: timestamp (nullable = true)


========== gold.fact_encounter ==========
Total records: 2483
Schema:
root
 |-- encounter_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- class_code: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- period_start: string (nullable = true)
 |-- period_end: string (nullable = true)
 |-- gold_loaded_at: timestamp (nullable = true)


========== gold.fact_observation ==========
Total records: 39309
Schema:
root
 |-- observation

In [14]:
# ============================================
# CELL 13: GOLD DATA QUALITY VALIDATION
# ============================================

checks = [
    ("Patient", "gold.dim_patient", "patient_id"),
    ("Encounter", "gold.fact_encounter", "encounter_id"),
    ("Observation", "gold.fact_observation", "observation_id"),
    ("Condition", "gold.fact_condition", "condition_id"),
    ("Date", "gold.dim_date", "date_key")
]

print("========== PRIMARY KEY VALIDATION ==========")

for name, table_name, key in checks:

    df = spark.table(table_name)

    duplicates = (
        df.groupBy(key)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    nulls = df.filter(F.col(key).isNull()).count()

    print(
        f"{name}: Duplicate keys = {duplicates}, "
        f"Null keys = {nulls}"
    )

print("\n========== FOREIGN KEY VALIDATION ==========")

patient_ids = (
    spark.table("gold.dim_patient")
    .select("patient_id")
    .distinct()
)

encounter_ids = (
    spark.table("gold.fact_encounter")
    .select("encounter_id")
    .distinct()
)

for table_name in [
    "gold.fact_encounter",
    "gold.fact_observation",
    "gold.fact_condition"
]:

    df = spark.table(table_name)

    missing_patients = (
        df.filter(F.col("patient_id").isNotNull())
        .join(patient_ids, "patient_id", "left_anti")
        .count()
    )

    print(
        f"{table_name}: Missing patient references = "
        f"{missing_patients}"
    )

for table_name in [
    "gold.fact_observation",
    "gold.fact_condition"
]:

    df = spark.table(table_name)

    missing_encounters = (
        df.filter(F.col("encounter_id").isNotNull())
        .join(encounter_ids, "encounter_id", "left_anti")
        .count()
    )

    print(
        f"{table_name}: Missing encounter references = "
        f"{missing_encounters}"
    )

print("\nGold data quality validation completed.")

StatementMeta(, fece9998-cbed-42df-ae4e-aaeaee85b9c8, 16, Finished, Available, Finished, False)

========== PRIMARY KEY VALIDATION ==========
Patient: Duplicate keys = 0, Null keys = 0
Encounter: Duplicate keys = 0, Null keys = 0
Observation: Duplicate keys = 0, Null keys = 0
Condition: Duplicate keys = 0, Null keys = 0
Date: Duplicate keys = 0, Null keys = 0

========== FOREIGN KEY VALIDATION ==========
gold.fact_encounter: Missing patient references = 0
gold.fact_observation: Missing patient references = 0
gold.fact_condition: Missing patient references = 0
gold.fact_observation: Missing encounter references = 0
gold.fact_condition: Missing encounter references = 1

Gold data quality validation completed.


In [15]:
# ============================================
# CELL 14: IDENTIFY MISSING ENCOUNTER REFERENCE
# ============================================

condition_df = spark.table("gold.fact_condition")

encounter_ids = (
    spark.table("gold.fact_encounter")
    .select("encounter_id")
    .distinct()
)

missing_encounter_records = (
    condition_df
    .filter(F.col("encounter_id").isNotNull())
    .join(
        encounter_ids,
        condition_df.encounter_id == encounter_ids.encounter_id,
        "left_anti"
    )
)

print("Conditions with missing encounter references:",
      missing_encounter_records.count())

display(missing_encounter_records)

StatementMeta(, fece9998-cbed-42df-ae4e-aaeaee85b9c8, 17, Finished, Available, Finished, False)

Conditions with missing encounter references: 1


SynapseWidget(Synapse.DataFrame, 6a5e112d-66f3-4fb6-b92f-6f9dd12a6d66)

In [16]:
# ============================================
# CELL 15: CHECK MISSING ENCOUNTER IN SILVER
# ============================================

print("Checking encounter ID: 208313")

silver_check = (
    spark.table("silver.encounter")
    .filter(F.col("id") == "208313")
)

print("Matching Silver records:", silver_check.count())

display(silver_check)

StatementMeta(, fece9998-cbed-42df-ae4e-aaeaee85b9c8, 18, Finished, Available, Finished, False)

Checking encounter ID: 208313
Matching Silver records: 0


SynapseWidget(Synapse.DataFrame, 8a59ec10-3cf0-4d62-b840-45f6ace5a791)

In [17]:
# ============================================
# CELL 16: VERIFY CONDITION IN SILVER
# ============================================

silver_condition_check = (
    spark.table("silver.condition")
    .filter(F.col("id") == "462259")
)

print("Matching Silver Condition records:",
      silver_condition_check.count())

display(silver_condition_check)

StatementMeta(, fece9998-cbed-42df-ae4e-aaeaee85b9c8, 19, Finished, Available, Finished, False)

Matching Silver Condition records: 2


SynapseWidget(Synapse.DataFrame, 1636c494-57be-4cc9-985c-1e3cc373dcf2)

In [18]:
# ============================================
# CELL 17: FINAL GOLD QUALITY SUMMARY
# ============================================

print("========== GOLD PIPELINE FINAL SUMMARY ==========")

tables = [
    "gold.dim_patient",
    "gold.fact_encounter",
    "gold.fact_observation",
    "gold.fact_condition",
    "gold.dim_date"
]

for table_name in tables:
    print(f"{table_name}: {spark.table(table_name).count()} records")

print("\n========== DATA QUALITY STATUS ==========")

print("Primary key validation: PASSED")
print("Duplicate key validation: PASSED")
print("Null key validation: PASSED")
print("Patient reference validation: PASSED")
print("Observation encounter reference validation: PASSED")
print("Condition encounter reference validation: 1 ORPHAN FOUND")

print("\nOrphan Condition ID: 462259")
print("Missing Encounter ID: 208313")
print("Root cause: Missing reference in Silver source data")
print("Action: Preserved source record; no deletion or dummy record")

print("\nGold pipeline execution completed.")

StatementMeta(, fece9998-cbed-42df-ae4e-aaeaee85b9c8, 20, Finished, Available, Finished, False)

========== GOLD PIPELINE FINAL SUMMARY ==========
gold.dim_patient: 1052 records
gold.fact_encounter: 2483 records
gold.fact_observation: 39309 records
gold.fact_condition: 1611 records
gold.dim_date: 30970 records

========== DATA QUALITY STATUS ==========
Primary key validation: PASSED
Duplicate key validation: PASSED
Null key validation: PASSED
Patient reference validation: PASSED
Observation encounter reference validation: PASSED
Condition encounter reference validation: 1 ORPHAN FOUND

Orphan Condition ID: 462259
Missing Encounter ID: 208313
Root cause: Missing reference in Silver source data
Action: Preserved source record; no deletion or dummy record

Gold pipeline execution completed.
